In [1]:
import pandas as pd
import numpy as np

In [2]:
import pandas as pd
import numpy as np

def create_SL_dataset(
    SL_pair: str,
    biomarker: str,
    target1: str,
    target2: str = None,
    gene_list_path: str = "hgnc_protein_coding_genes",
    hgnc_path: str = "/Users/metinyazar/Desktop/drug_resistance_ppi/database_files/hgnc/hgnc_complete_set.txt",
    output_path: str = None
):
    """
    Create dataset for predicting SL resistance features.

    Parameters
    ----------
    SL_pair : str
        Synthetic lethal pair name (e.g. "MSH6_WRN").
    biomarker : str
        Biomarker gene symbol (e.g. "MSH6").
    target1 : str
        Primary target gene symbol (e.g. "WRN").
    target2 : str, optional
        Secondary target gene symbol (default None).
    gene_list_path : str
        Path to HGNC protein coding gene list file.
    hgnc_path : str
        Path to full HGNC complete set file.
    output_path : str, optional
        Path to save the output CSV. If None, no file is saved.

    Returns
    -------
    pd.DataFrame
        Final dataset.
    """

    # === Load gene list ===
    gene_list = pd.read_csv(gene_list_path, sep="\t")
    main = gene_list[['symbol', 'entrez_id','hgnc_id' ,'ensembl_gene_id']].copy()
    main = main.rename(columns={
        'symbol': 'Query',
        'hgnc_id': 'hgnc_id_query',
        'ensembl_gene_id': 'ensembl_gene_id_query',
        'entrez_id': 'entrez_id_query'
    })
    main['Screen'] = "Clinical_Trials"
    main['SL_Pair'] = SL_pair
    main['Biomarker'] = biomarker
    main['Target1'] = target1
    main['Target2'] = np.nan if target2 is None else target2
    main = main[~main["Query"].isin([target1, biomarker] if target2 is None else [target1, biomarker, target2])]

    # === HGNC lookup ===
    hgnc = pd.read_csv(hgnc_path, sep="\t")
    columns_to_include = ['hgnc_id', 'symbol','prev_symbol','ensembl_gene_id','alias_symbol','entrez_id']
    hgnc_dict = hgnc[columns_to_include].to_dict(orient='records')

    gene_lookup = {}

    def normalize_symbol(s):
        return s.strip().upper() if isinstance(s, str) else None

    for entry in hgnc_dict:
        hgnc_id = entry.get('hgnc_id', np.nan)
        ensembl_id = entry.get('ensembl_gene_id', np.nan)
        entrez_id = str(entry.get('entrez_id', 'NA')) if pd.notnull(entry.get('entrez_id')) else 'NA'

        # Primary
        symbol = normalize_symbol(entry.get('symbol'))
        if symbol:
            gene_lookup[symbol] = (hgnc_id, ensembl_id, entrez_id)

        # Previous
        if pd.notnull(entry.get('prev_symbol')):
            for prev_symbol in entry['prev_symbol'].split('|'):
                norm_prev = normalize_symbol(prev_symbol)
                if norm_prev and norm_prev not in gene_lookup:
                    gene_lookup[norm_prev] = (hgnc_id, ensembl_id, entrez_id)

        # Aliases
        if pd.notnull(entry.get('alias_symbol')):
            for alias in entry['alias_symbol'].split('|'):
                norm_alias = normalize_symbol(alias)
                if norm_alias and norm_alias not in gene_lookup:
                    gene_lookup[norm_alias] = (hgnc_id, ensembl_id, entrez_id)

    def lookup_gene_info(gene_symbol):
        key = normalize_symbol(gene_symbol)
        return gene_lookup.get(key, (np.nan, np.nan, 'NA'))

    # === Lookup IDs ===
    main['hgnc_id_biomarker'], main['ensembl_gene_id_biomarker'], main['entrez_id_biomarker'] = zip(
        *main['Biomarker'].apply(lookup_gene_info)
    )
    main['hgnc_id_target1'], main['ensembl_gene_id_target1'], main['entrez_id_target1'] = zip(
        *main['Target1'].apply(lookup_gene_info)
    )
    main['hgnc_id_target2'], main['ensembl_gene_id_target2'], main['entrez_id_target2'] = zip(
        *main['Target2'].apply(lookup_gene_info)
    )

    # Force entrez_id to string
    for col in ["entrez_id_biomarker", "entrez_id_target1", "entrez_id_target2"]:
        main[col] = main[col].astype(str)

    # === Reorder columns ===
    column_order = (
        ["Screen", "SL_Pair", "Query", "Biomarker", "Target1", "Target2",
         "entrez_id_query", "hgnc_id_query", "ensembl_gene_id_query",
         "entrez_id_biomarker", "hgnc_id_biomarker", "ensembl_gene_id_biomarker",
         "entrez_id_target1", "hgnc_id_target1", "ensembl_gene_id_target1",
         "entrez_id_target2", "hgnc_id_target2", "ensembl_gene_id_target2"] +
        [col for col in main.columns if col not in [
            "Screen", "SL_Pair", "Query", "entrez_id_query", "hgnc_id_query", "ensembl_gene_id_query",
            "Biomarker", "Target1", "Target2",
            "entrez_id_biomarker","hgnc_id_biomarker","ensembl_gene_id_biomarker",
            "entrez_id_target1","hgnc_id_target1","ensembl_gene_id_target1",
            "entrez_id_target2","hgnc_id_target2","ensembl_gene_id_target2"
        ]]
    )
    main = main[column_order]

    # === Add placeholder feature columns ===
    feature_cols = [
        "StringInteractionWithBiomarker", "StringInteractionWithTarget",
        "CoexpressionWithBiomarker", "CoexpressionWithTarget",
        "AvgExpression", "CoessentialityWithBiomarker", "CoessentialityWithTarget",
        "FET_SharedInteractors_Biomarker_BIOGRID", "FET_SharedInteractors_Target_BIOGRID",
        "FET_SharedInteractors_Biomarker_STRING", "FET_SharedInteractors_Target_STRING"
    ]
    for col in feature_cols:
        main[col] = None

    # === Save if requested ===
    if output_path:
        main.to_csv(output_path, index=False)

    return main


In [3]:
import pandas as pd

# Read your Excel file
pairs_df = pd.read_excel("biomarker_target_genes.xlsx")  
# Expected columns: "Biomarker", "Target1" (and optionally "Target2")

# Loop through rows and generate datasets
for _, row in pairs_df.iterrows():
    biomarker = row["Biomarker"]
    target1 = row["Target"]
    target2 = row.get("Target2", None)  # if no Target2 column, it will stay None
    
    SL_pair = f"{biomarker}_{target1}" if pd.isna(target2) else f"{biomarker}_{target1}_{target2}"
    
    output_path = f"datasets/PredictingSLResistanceFeatures_{SL_pair}_main.csv"
    
    df = create_SL_dataset(
        SL_pair=SL_pair,
        biomarker=biomarker,
        target1=target1,
        target2=target2,
        output_path=output_path
    )
    
    print(f"✅ Created dataset for {SL_pair} → {output_path}")


/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_13404/3518474690.py:40: DtypeWarning: Columns (35,38,45,48,49) have mixed types. Specify dtype option on import or set low_memory=False.
  gene_list = pd.read_csv(gene_list_path, sep="\t")
/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_13404/3518474690.py:56: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc = pd.read_csv(hgnc_path, sep="\t")


✅ Created dataset for MSH6_WRN → datasets/PredictingSLResistanceFeatures_MSH6_WRN_main.csv


/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_13404/3518474690.py:40: DtypeWarning: Columns (35,38,45,48,49) have mixed types. Specify dtype option on import or set low_memory=False.
  gene_list = pd.read_csv(gene_list_path, sep="\t")
/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_13404/3518474690.py:56: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc = pd.read_csv(hgnc_path, sep="\t")


✅ Created dataset for CDH1_ROS1 → datasets/PredictingSLResistanceFeatures_CDH1_ROS1_main.csv


/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_13404/3518474690.py:40: DtypeWarning: Columns (35,38,45,48,49) have mixed types. Specify dtype option on import or set low_memory=False.
  gene_list = pd.read_csv(gene_list_path, sep="\t")
/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_13404/3518474690.py:56: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc = pd.read_csv(hgnc_path, sep="\t")


...

✅ Created dataset for BRCA1_POLQ → datasets/PredictingSLResistanceFeatures_BRCA1_POLQ_main.csv


/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_13404/3518474690.py:40: DtypeWarning: Columns (35,38,45,48,49) have mixed types. Specify dtype option on import or set low_memory=False.
  gene_list = pd.read_csv(gene_list_path, sep="\t")
/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_13404/3518474690.py:56: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc = pd.read_csv(hgnc_path, sep="\t")


✅ Created dataset for KRAS_PLK1 → datasets/PredictingSLResistanceFeatures_KRAS_PLK1_main.csv


/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_13404/3518474690.py:40: DtypeWarning: Columns (35,38,45,48,49) have mixed types. Specify dtype option on import or set low_memory=False.
  gene_list = pd.read_csv(gene_list_path, sep="\t")
/var/folders/cs/dzdz0ppx7lld40ztphtqg_dh0000gp/T/ipykernel_13404/3518474690.py:56: DtypeWarning: Columns (32,34,38,40,50) have mixed types. Specify dtype option on import or set low_memory=False.
  hgnc = pd.read_csv(hgnc_path, sep="\t")


✅ Created dataset for MTAP_PRMT5 → datasets/PredictingSLResistanceFeatures_MTAP_PRMT5_main.csv
